In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-


# Deep Neural Networks

## Session 04
### Neural Network with :
- One hidden layer 
- ${Tanh}$ activation function
- Experiment with EPOCHs, ALPHA and Random State


<img src='../../prasami_images/prasami_color_tutorials_small.png' width='400' alt="By Pramod Sharma : pramod.sharma@prasami.com"/>

In [ ]:
###-----------------
### Import Libraries
###-----------------

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from collections.abc import Callable
from typing import Literal

from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, ConfusionMatrixDisplay

%matplotlib inline

In [ ]:
###----------------
### Some parameters
###----------------

inpDir = '../input'
outDir = '../output'

RANDOM_STATE = 24 # REMEMBER: to remove at the time of promotion to production
np.random.seed(RANDOM_STATE) # Set Random Seed for reproducible  results
rng = np.random.default_rng(seed = RANDOM_STATE) 

EPOCHS = 20001 # number of epochs
ALPHA = 0.1 # learning rate
NUM_SAMPLES = 1000 # How many samples we want to generate 
NOISE = 0.2 # Noise to be introduced in the data
TEST_SIZE = 0.2

# parameters for Matplotlib
params = {'legend.fontsize': 'large',
          'figure.figsize': (15, 6),
          'axes.labelsize': 'large',
          'axes.titlesize':'large',
          'xtick.labelsize':'medium',
          'ytick.labelsize':'medium'
         }

CMAP = 'coolwarm' # plt.cm.Spectral

plt.rcParams.update(params)

### Helper Functions

In [ ]:
###-----------------------------------
### Function to plot Decision boundary
###-----------------------------------

def fn_plot_decision_boundary( model: dict, X_tr: np.ndarray, y_tr: np.ndarray, X_ts :  np.ndarray, y_ts:  np.ndarray,):
    '''
        Args:
            model: trained neural network model
            X_tr : train feature matrix
            y_tr : train labels
            X_ts : test feature matrix
            y_ts : test labels
       Return:
           None
    '''
    
    # grid size for mesh grid
    h = 0.01
    margin = 10* h

    # Set min and max values and give it some padding
    xMin = min(X_tr[:, 0].min() - margin, X_ts[:, 0].min() - margin)
    xMax = max(X_tr[:, 0].max() + margin, X_ts[:, 0].max() + margin)
    yMin = min(X_tr[:, 1].min() - margin, X_ts[:, 1].min() - margin)
    yMax = max(X_tr[:, 1].max() + margin, X_ts[:, 1].max() + margin)
    
    # Generate a grid of points with distance 'h' between them
    xx, yy = np.meshgrid(np.arange(xMin, xMax, h), np.arange(yMin, yMax, h))
    
    # Predict the function value for the whole grid
    Z = fn_predict(model, np.c_[xx.ravel(), yy.ravel()])
    
    # Make its shape same as that of xx 
    Z = Z.reshape(xx.shape)

    fig = plt.figure()
    ax = fig.add_axes(111)
    
    # Now we have Z value corresponding to each of the combination of xx and yy
    # Plot the contour and training examples
    contour =  ax.contourf(xx, yy, Z, cmap=CMAP) #, alpha = 0.8
    
    # Plotting scatter for train data
    ax.scatter(X_tr[:, 0], X_tr[:, 1], c=y_tr,
                                  s=30, edgecolor='k', cmap=plt.cm.coolwarm, label='Train')
    
    
    # Plotting scatter for test data
    ax.scatter(X_ts[:, 0], X_ts[:, 1], c=y_ts,
                                  s=150, marker = '*',edgecolor='k', cmap=plt.cm.inferno, label='Test')

    #Add labels and legend
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')
    ax.set_title('Decision Boundary')
    ax.legend()
    
    # Add colorbar for the decision boundary
    plt.colorbar(contour, ax=ax, label='Prediction Score')
    
    plt.tight_layout()
    plt.show()

## Generate Data Set
<p style="font-family: Arial; font-size:1.2em;">
Use Sklearn's dataset generator <a href="http://scikit-learn.org/stable/modules/generated/sklearn.datasets.make_moons.html">make_moon</a>.
</p>

In [ ]:
X, y = datasets.make_moons(n_samples=NUM_SAMPLES, shuffle=True, noise=NOISE, random_state=RANDOM_STATE)

In [ ]:
# Lets Plot the data
plt.scatter(X[:,0], X[:,1], s=40, c=y, marker='*', cmap=CMAP)

plt.grid()

In [ ]:
#  Split the data in training and test sets to measure performance of the model.
X_train, X_test, y_train, y_test = train_test_split(X, y, 
                                                    test_size=TEST_SIZE,
                                                    random_state=RANDOM_STATE )

print (X_train.shape, y_train.shape, X_test.shape, y_test.shape)

In [ ]:
# if you really want to save on space, convert to float32

X_train = X_train.astype(np.float32)

X_test = X_test.astype(np.float32)

y_train = y_train.astype(np.float32)

y_test = y_test.astype(np.float32)

## Single neuron
<img src="../../images/dnn_nb_s03_fig1.jpg" width='400' align = 'left'>

## Single Perceptron
<img src="../../images/dnn_nb_s03_fig2.jpg" width='400' align = 'left'>

### For single row of perceptrons:
<img src="../../images/dnn_nb_s04_fig3.png" width='800' align = 'left' alt = 'Two in a row'>

### For single perceptron:
   
$
\begin{aligned}
a &=  \sigma(z)\\
a &=  \sigma(x_1.w_1 + x_2.w_2 + b)\\ 
a &= \sigma\ ( [ x_1, x_2 ] \circ
\begin{bmatrix} w_1 \\ w_2 \end{bmatrix}  + b )\\
\end{aligned}
$

#### For multiple Rows of X:

$
\begin{aligned}
a &= \sigma\ (\begin{bmatrix} x_1^{(1)} & x_2^{(1)}\\ 
x_1^{(2)} & x_2^{(2)}\\
x_1^{(...)} & x_2^{(...)}\\
x_1^{(m)} & x_2^{(m)} \end{bmatrix} \circ
\begin{bmatrix} w_1 \\ w_2 \end{bmatrix}  + b )\\
\end{aligned}
$

In matrix form it can be represented as:

$
\begin{aligned}
a &= \sigma\ ( X_{shape = (m,2)} \circ W_{shape = (2,1)}^{[1]} + b_{shape = (1,1)})
\end{aligned}
$

**Note:** Please note that Python is going to broadcast b in all $'m'$ rows. Avoid any confusion, always maintain dimensions of $b$.

### For single row of perceptrons:

$
\begin{aligned}
a^{[1]} &= g ( X_{shape = (m,2)} \circ W_{shape = (2,1)}^{[1]} + b_{shape = (1,1)}^{[1]})\\
a^{[2]} &= \sigma ( a^{[1]}_{shape = (m,1)} \circ W_{shape = (1,1)}^{[2]} + b_{shape = (1,1)}^{[2]})\\
Cost &= \text{Loss function} \\
&= - \large[y \circ log_e (a^{[2]}) + (1-y) \circ log_e (1- a^{[2]})]\\
\end{aligned}
$

We will be using $\tanh$ function for layer 1 (hidden layer) as it fits in majority of cases and its derivative can simply be represented as 1 -$\tanh^2(z_1)$. Since our output is binary, it makes sense to use $\text{Sigmoid}$ in the last layer.

## Neural Network

Let's start with simple network. Our data has **two** features. Hence size of input layer will also be two. The output is binary, we can code it as single column as well as double column output. The hidden layer could be of **any size**. One need to execute a handful of iterations to arrive at right size of hidden layer. For purpose of today's discussions, size of hidden layer is taken as shown below.

<img src='../../images/dnn_nb_s04_fig1.jpg' width = '500'/>

<img src='../../images/dnn_nb_s04_fig2.png' width = '500'/>

## What is this hidden layer?

<img src='../../images/dnn_nb_s04_fig5.jpg' width = '500'/>

## Forward Propagation

### For single Neuron:
$\text{activations (a)} = \text{activation function} ( X \circ W_1 + b)$

Hence for hidden layer, we can write as follows:

$
\begin{aligned}
z_1^{[1]} & = X . W_1^{[1]} + b_1^{[1]}\\
a_1^{[1]} & = \tanh(z_1^{[1]}) \\
\\
z_2^{[1]} & = X . W_2^{[1]} + b_2^{[1]} \\
a_2^{[1]} & = \tanh(z_2^{[1]}) \\
\\
z_3^{[1]} & = X . W_3^{[1]} + b_3^{[1]} \\
a_3^{[1]} & = \tanh(z_3^{[1]}) \\
\\
z_4^{[1]} & = X . W_4^{[1]} + b_4^{[1]} \\
a_4^{[1]} & = \tanh(z_4^{[1]}) \\
\\
\text{Or}\\
a^{[1]} &= \tanh(X \circ \begin{bmatrix} W_1^{[1]}, &W_2^{[1]}, &W_3^{[1]}, &W_4^{[1]}\end{bmatrix} + \begin{bmatrix}b_1^{[1]}, &b_2^{[1]}, &b_3^{[1]}, &b_4^{[1]}\end{bmatrix} )\\
\end{aligned}
$



<hr>

If we convert above to matrix version, we can say.

$
\begin{aligned}
\text{Replace:}\\
W^{[1]} & = \begin{bmatrix} W_1^{[1]}, &W_2^{[1]}, &W_3^{[1]}, &W_4^{[1]}\end{bmatrix}\\
b^{[1]} & = \begin{bmatrix}b_1^{[1]}, &b_2^{[1]}, &b_3^{[1]}, &b_4^{[1]}\end{bmatrix}\\
\text{Hence:}
\\z_{shape = (m,4)}^{[1]} & = X_{shape = (m,2)} \circ W_{shape=(2,4)}^{[1]} + b_{shape = (1,4)}^{[1]} \\
\\
a_{shape = (m,4)}^{[1]} & = \tanh(z^{[1]}) \\
\end{aligned}
$

Similarly for second layer.

$
\begin{aligned}
z_{shape = (m, 1)}^{[2]} & = a_{shape = (m,4)}^{[1]} \circ W_{shape=(4,1)}^{[2]} + b_{shape = (1,1)}^{[2]} \\
\\
a_{shape = (m, 1)}^{[2]} & = \hat{y} = \mathrm{sigmoid}(z^{[2]})\\
\end{aligned}
$

Where:

Sigmoid $\sigma$: 

$\sigma(z) = \dfrac{1}{1 + e^{-z}}$

## Activation Functions

### Sigmoid Function

In [ ]:
# sigmoid function is applied to each Value independently
def fn_sigmoid(z : np.ndarray)-> np.ndarray:
    '''
    Args:
        z : a matrix of z values of shape (m, n_output)
    returns:
        sigmoid values of z
    
    '''
    
    return 1 / ( 1 + np.exp ( -z ) )
    

In [ ]:
sm = fn_sigmoid(np.asarray([[-1, 0., 1.], [-np.inf, 0., np.inf]]))
print (sm)

### Tanh Activation Function

In [ ]:
def fn_activ(z: np.ndarray) -> np.ndarray:
    
    '''
        Args:
           z : array, Aggregated values 
       Return:
           Activations for each z
    '''

    return np.tanh(z)

def fn_activ_prime(z: np.ndarray) -> np.ndarray:
    '''
        Args:
           z : array, Activation
       Return:
           Derivative, for each z
    '''

    return 1.0 - np.tanh(z)**2
    #return 1.0 - a**2


In [ ]:
fn_activ(np.asarray([[-1, 0., 1.], [-np.inf, 0., np.inf]]))

### Is our Activation Function working?

In [ ]:
np.tanh(0.5)

In [ ]:
(1 - np.power(np.tanh(0.5), 2))

In [ ]:
fn_activ_prime(0.5)

## Predict Function

For predictions, we will simply be using the forward propagation. No need to iterate or calculate the back propagation for supervised learning.


In [ ]:
# Helper function to predict an output (0 or 1)

def fn_predict(model : dict, X: np.ndarray) -> np.ndarray:
    '''
     Args:
         model
         X: input features
    Returns:
        Predictions against the instances
         
    '''
    W1, b1, W2, b2 = model['W1'],model['b1'],model['W2'],model['b2']
    
    #### Forward Propagation   
    # Layer 1
    z1 = X.dot(W1) + b1 # Aggregation
    a1 = fn_activ (z1) # Activation

    # Layer 2
    z2 = a1.dot(W2) + b2
    a2 = fn_sigmoid(z2)
    
    return a2>=0.5 # Is it greater than 0.5?

## Loss Function

We need to minimize the error by adjusting ($W_s, b_s$). We call the function that measures our error the <b>loss function</b>. A common choice with the sigmoid output is the cross-entropy loss. The loss for predictions $\hat{y}$ with respect to the true labels $y$ is given by:

$
\begin{aligned}
L(\hat{y_i}, y_i) =  -\begin{bmatrix}y_i.log(\hat{y_i}) + (1 - y_i) . log(1-\hat{y_i})\end{bmatrix}\\
\end{aligned}
$

### Remember
<b>In binary classification, why are we calculating log(1-ŷ)?</b>

Because in binary classification with a single sigmoid neuron:
- The neuron only predicts probability of class 1
- When y=0, we want to penalize high predictions of class 1
- $-log(1−\hat{y})$ does exactly this - it's high when $\hat{y}$ is high (wrong) and low when $\hat{y}$ is low (correct).

It's NOT because "other true label is 0" - it's because the prediction $\hat{y}$ represents probability of class 1, and we need a way to measure error when the truth is class 0!

<b>For all samples:</b>

$
\begin{aligned}
J(\hat{y}, y) =  -\frac{1}{m}\sum_{i=1}^{m}\begin{bmatrix}y_i.log(\hat{y}_i) + (1-y_i) . log(1-\hat{y}_i)\end{bmatrix}\\
\end{aligned}
$


We can use gradient descent to find its minimum.

In [ ]:
# function to evaluate the total loss on the dataset

def fn_calculate_loss(model : dict, X_l: np.ndarray, y_l: np.ndarray) -> np.float64:
    '''
    Args:
        model: dictionay object containing weights and biases
        X_l: Feature Matrix
        y_l: Labels array
    Returns:
        Average loss
    '''
    m = X_l.shape[0]

    W1, b1, W2, b2 = model['W1'],model['b1'],model['W2'],model['b2']
    

    #### Forward Propagation   
    # Layer 1
    z1 = X_l.dot(W1) + b1 # Aggregation
    a1 = fn_activ (z1) # Activation

    # Layer 2
    z2 = a1.dot(W2) + b2
    a2 = fn_sigmoid(z2)
    
    data_loss = -(y_l* np.log(a2) + (1-y_l)* (np.log(1-a2))).sum()

    return data_loss / m

## For a single row of data x

<img src='../../images/dnn_nb_s04_fig4.png' style='width: 800px;'/>

As an input, gradient descent needs the gradients (vector of derivatives) of the loss function with respect to our parameters: 

$\frac{\partial{L}}{\partial{W_1}}(= \partial{W^{[1]}})$, $\frac{\partial{L}}{\partial{b_1}}(= \partial{b^{[1]}})$, $\frac{\partial{L}}{\partial{W_2}}(= \partial{W^{[2]}})$, $\frac{\partial{L}}{\partial{b_2}}(= \partial{b^{[2]}})$. 

To calculate these gradients we use the <b>back-propagation algorithm</b>.

**Note:** Loss is a function of $a^{[2]}$ which is a function of $z^{[2]}$ and so on. It can be further represented as follows:<br>
$
\begin{aligned}
Loss &= f_1(a^{[2]})\\
a^{[2]} &= f_2(z^{[2]})\\
z^{[2]} &= f_3(a^{[1]})\\
a^{[1]} &= f_4(z^{[1]})\\
z^{[1]} &= f_5(X)\\
\text{Therefore:}\\
\frac{\partial{Loss}}{\partial{z^{[1]}}} &= \frac{\partial{Loss}}{\partial{z^{[2]}}} \circ
\frac{\partial{z^{[2]}}}{\partial{a^{[1]}}} \circ \frac{\partial{a^{[1]}}}{\partial{z^{[1]}}} 
\end{aligned}
$

## Back-propagation for all Rows
For all rows, equations will remain same and the values will be divided by <b><i>'m'</i></b>; number of samples.

$
\begin{aligned}
\partial{z^{[2]}}  & = a^{[2]} - y  \\
\partial{W^{[2]}}  & = \frac{1}{m} a^{[1]T}\circ \partial{z^{[2]}} \\
\partial{b^{[2]}}  & = \frac{1}{m} \mathrm{np.sum}(\partial{z^{[2]}}, axis = 0, keepdims = True) \\
\partial{a^{[1]}}  & = \partial{z^{[2]}} \circ W^{[2]T}\\
\\
\partial{z^{[1]}}  & = \partial{a^{[1]}} * ( 1-tanh(z^{[1]})^2)\\
\partial{W^{[1]}}  & = \frac{1}{m} X^{T}\circ \partial{z^{[1]}} \\
\partial{b^{[1]}}  & = \frac{1}{m} \mathrm{np.sum}(\partial{z^{[1]}}, axis = 0, keepdims = True) \\
\\
\end{aligned}
$

## Notes:


We have transposed a few matrices in above calculations such as $a^{[1]}$, $W^{[2]}$ and X. A review of shapes of matrices will reveal that this adjustment is needed to have consistent sizes. e.g.

- Shape of $a^{[1]}$ and $\partial{z}^{[2]}$ are ( m, 4) and ( m, 1 ) respectively. Expected shape of $\partial{W^{[2]}}$ is ( 4, 1 ) which is same as that of $W^{[2]}$.
- In equation $\partial{z^{[1]}}  = \partial{z^{[2]}}\circ  W^{[2]T} * (  1-tanh(z^{[1]})^2)$ shape of $z^{[2]}$,  $W^{[2]}$ and $a^{[1]}$ are (m,1), (4,1) and (m,4). For element wise multiplication, expected shape of dot product of is $z^{[2]}$ and $W^{[2]}$ is ( m, 4 ).
- partial differentiation of Tanh is $(1-Tanh^2)$. a = tanh(z) amd hence 
- Lastly, shape of $\partial{W^{[1]}}$ is (2,4) and that of X and $\partial{z^{[1]}}$ are ( m, 2 ) and ( m, 4 ).

## Model

In [ ]:
# prepare the Model

def fn_build_model(nn_hdim : np.int64, 
                X : np.ndarray, 
                y : np.ndarray, 
                epochs: np.int64 = EPOCHS, 
                alpha: np.float64 = ALPHA) -> dict:
    
    '''
    Args:
        nn_hdim : Number of nodes in the hidden layer
        X : np.ndarray; Training features to train
        y : np.ndarray; Training targets (labels)
        epochs : Number of passes through the training data for gradient descent
        alpha : learning rate
        
    Returns:
        Model: Dictionary object containing weights and biases
    '''
    
    # Initialize the parameters to random values. We need to learn these Weights
    
    '''
    ##### 
            Change from Rand to Randn gives different shape of the loss curve
    #####
    '''
    m, nn_input_dim = X.shape # training set size (rows and cols)
    nn_output_dim = y.shape[1] # output layer dimensionality

    W1 = rng.random((nn_input_dim, nn_hdim), dtype=np.float32) / np.sqrt(nn_input_dim)
    W2 = rng.random((nn_hdim, nn_output_dim), dtype=np.float32) / np.sqrt(nn_hdim)
    
    b1 = np.zeros ((1, nn_hdim), dtype=np.float32)
    b2 = np.zeros ((1, nn_output_dim), dtype=np.float32)
    
    curr_loss = 0
    
    loss = []
    epoch = []
    
    for i in range (0, epochs):
        
        #### Forward Propagation
        
        # Layer 1
        z1 = X.dot(W1) + b1 # Aggregation
        a1 = fn_activ (z1)  # Activation
        
        # Layer 2
        z2 = a1.dot(W2) + b2 # Aggregation
        a2 = fn_sigmoid(z2)  # Activation
        
        ### Back Propagation

        # Layer 2
        dz2 = a2 - y
        
        dW2 = (a1.T).dot(dz2)
        assert (W2.shape == dW2.shape), 'Shape of W2 {} and shape of dW2 {}'.format(W2.shape, dW2.shape)
        
        db2 = np.sum(dz2, axis = 0, keepdims=True)
        assert (b2.shape == db2.shape), 'Shape of b2 {} and shape of db2 {}'.format(b2.shape, db2.shape)
        
        da1 = dz2.dot(W2.T)
        assert (a1.shape == da1.shape), 'Shape of a1 {} and shape of da1 {}'.format(a1.shape, da1.shape)

        # Layer 1
        dz1 = da1 * fn_activ_prime(z1)
        assert (z1.shape == dz1.shape), 'Shape of z1 {} and shape of dz1 {}'.format(z1.shape, dz1.shape)

        dW1 = (X.T).dot(dz1)
        assert (W1.shape == dW1.shape), 'Shape of W1 {} and shape of dW1 {}'.format(W1.shape, dW1.shape)
        db1 = np.sum(dz1, axis = 0, keepdims=True)
        assert (b1.shape == db1.shape), 'Shape of b1 {} and shape of db1 {}'.format(b1.shape, db1.shape)

        ########################
        ### Gradient Updates ###
        ########################
        # gradients are being updated for every epoch
        W1 = W1 - alpha * dW1 / m
        W2 = W2 - alpha * dW2 / m
        b1 = b1 - alpha * db1 / m
        b2 = b2 - alpha * db2 / m
        
        # Store model in a dict object
        model = {'W1': W1, 'b1': b1, 'W2': W2, 'b2': b2}
        
        # for every nth epoch calculate loss for future plotting
        if i%100 ==  0:
            curr_loss = fn_calculate_loss (model, X, y)
            loss.append(curr_loss)
            epoch.append(i)
            
        # print loss every nth epoch    
        if i%1000 == 0:
            print ('loss after {:>5d} epochs : {:.5f}'.format(i, curr_loss))
    
    # update loss_hist
    loss_hist['epoch'] =epoch
    loss_hist['loss'] = loss
    
    
    return model

In [ ]:
# lists to facilitate plotting 
loss_hist = {}

**Note:** - The vector y_train is being converted to 2-dimensional array

In [ ]:
# y_train must be (m,1) not (m,) for matrix multiplication in backprop

y_train = y_train.reshape(-1,1)
y_train.shape

In [ ]:
# Build a model with a 4-dimensional hidden layer
model = fn_build_model(4, X_train, y_train,
                    epochs = EPOCHS, 
                    alpha = ALPHA)

In [ ]:
model

Would contest that we should have used higher epochs as loss is still coming down. 

### Question
- How many epochs are sufficient?

## Make predictions

In [ ]:
y_pred = fn_predict(model, X_train)
print('Accuracy score on Train Data :', accuracy_score(y_train, y_pred))

In [ ]:
print(classification_report(y_train, y_pred))

In [ ]:
y_pred = fn_predict(model, X_test)

print('Accuracy score on Test Data :', accuracy_score(y_test, y_pred))

In [ ]:
print(classification_report(y_test, y_pred))

## Plot loss and decision boundary

In [ ]:
loss_df = pd.DataFrame(loss_hist)

fn_plot_decision_boundary(model, X_train, y_train, X_test, y_test) # plot decision boundary for this plot

## How weights mature!
- See on the video in the output directory.

In [ ]:
fig, ax = plt.subplots()

loss_df.plot(x = 'epoch', y = 'loss', ax = ax)

# little beautification
txtstr = "Errors: \n  Start : {:7.4f}\n   End : {:7.4f}".format(loss_df.iloc[0]['loss'],
                                                                loss_df.iloc[-1]['loss']) #text to plot

# properties  matplotlib.patch.Patch 
props = dict(boxstyle='round', facecolor='aqua', alpha=0.5)

# place a text box in upper left in axes coords

ax.text(0.6, 0.95, txtstr, transform=ax.transAxes, fontsize=14,
        verticalalignment='top', bbox=props)

ax.set_xlabel("Epochs")
ax.set_ylabel("Error")
ax.set_title('Overall')
ax.grid(True);
plt.tight_layout()

In [ ]:
cm  = confusion_matrix(y_test, y_pred)
cm

In [ ]:
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                              display_labels=[0,1])

fig, ax = plt.subplots(figsize = (4,4))

disp.plot(ax = ax, cmap = 'Blues', colorbar=False)

plt.show();